# Lab 2: Logistic Regression and SVM with Olink Proteomics Data

In this lab, we'll build binary classifiers from plasma Olink proteomics data in melanoma patients treated with PD-1 blockade.
Our goal is to predict whether a patient is a **responder** (`R`) or **non-responder** (`NR`) at baseline.

We'll use two openly available cohorts from the same study:

- Training cohort: `raw_training_data.csv`
- Validation cohort: `raw_validation_data.csv` (*Note that although this is called the "validation cohort," it corresponds to our test set*)

This gives us a realistic setup where we build our model on one cohort and evaluate on an external cohort.
The data are from: Mehta A, Rucevic M, Sprecher E, et al. *Plasma proteomic biomarkers identify non-responders and reveal biological insights about the tumor microenvironment in melanoma patients after PD1 blockade*. bioRxiv (2022). https://doi.org/10.1101/2022.02.02.478819

This will also illustrate one of the most common challenges in the field: a lack of data to answer the questions we want to model.
You'll see in the end that this makes it very difficult to make good modeling choices, especially choices that are supported by new data to which the model is applied.


In [ ]:
import matplotlib.pyplot as plt  # Plotting
import pandas as pd  # Working with tabular data
import requests  # Needed to help us get data from the web.
import seaborn as sns  # Make plotting prettier / easier

# scikit-learn is the standard Python package for many
# common machine learning algorithms
from sklearn.linear_model import LogisticRegression

# These metrics will help us know how well we did
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
)

# This function will help us split data:
from sklearn.model_selection import train_test_split

# scikit-learn provides utilities to make building
# ML pipelines easier.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC  # "Support Vector Classifier"

## Load and preprocess the data

The data for this preprint are stored on Mendeley Data.
Here, we wrote a custom function to retrieve the data from the Mendeley Data API.


In [ ]:
def load_data():
    """Retrieve the URLs for the training and test splits."""
    base_url = "https://data.mendeley.com/public-api/datasets/5kbzn5nv8x/files"
    files = requests.get(
        base_url,
        params={"version": 2, "folder_id": "root"},
        timeout=30,
    ).json()

    out = []
    for prefix in ["training", "validation"]:
        filename = f"raw_{prefix}_data.csv"
        file_match = [f for f in files if f["filename"] == filename]
        file_id = file_match[0]["id"]
        download_info = requests.get(
            f"{base_url}/{file_id}/file_downloaded",
            params={"version": 2},
            timeout=30,
        ).json()
        out.append(pd.read_csv(download_info["url"]))

    return out


train_long, test_long = load_data()

print("Training (long) shape:", train_long.shape)
print("Validation (long) shape:", test_long.shape)

display(train_long.head())
display(test_long.head())

Now that we have the data loaded, we need to transform them for modeling.
Our first step is to define our cohorts and target variable.
In this case, we've chosen to restrict the cohorts to only samples collected prior to any treatment with checkpoint inhibitors.

For the target variable, we then recode the `response_simple` column to be numeric, where `1` is our positive class (responders) and `0` is our negative class (non-responders), so:
- `1` = responder (`R`)
- `0` = non-responder (`NR`)

These tables are in long format, such that each row contains a single protein-sample pair.
For modeling, we need to pivot this table to look like a matrix, where each protein is a column.


In [ ]:
def prepare_data(split):
    """Preprocess the data for modeling."""
    # Keep only the "Baseline" time points or pre-treatment time points
    keep = split["Timepoint"] == "Baseline"
    if "OnTr" in split.columns:
        keep = keep | (split["OnTr"] == "pre")

    split_filtered = split.loc[keep].copy()

    # Transform "R" and "NR" into 1s and 0s:
    label_map = {"R": 1, "NR": 0}
    split_filtered["target"] = (
        split_filtered["response_simple"].map(label_map).astype(int)
    )

    # Pivot the long table to a wide one:
    split_wide = split_filtered.pivot_table(
        index=["SampleId", "target"],
        columns="Assay",
        values="NPX",
        aggfunc="mean",
    )

    return split_wide, split_wide.index.get_level_values("target")


# Create the training and test sets:
train_wide, train_labels = prepare_data(train_long)
test_wide, test_labels = prepare_data(test_long)

print("Training set baseline samples:", train_wide.shape[0])
print("Test set baseline samples:", test_wide.shape[0])
print("Training set class counts:")
print(train_labels.value_counts().sort_index())
print("\nTest set class counts:")
print(test_labels.value_counts().sort_index())

The two cohorts do not always have identical assay panels.
To make a fair external test set, we keep only shared proteins that were measured in both cohorts.

We also impute any missing values using their median value.
This may not be the best choice, but it will do for now.


In [ ]:
# Keep assays measured in both cohorts.
feature_cols = sorted(set(train_wide.columns).intersection(test_wide.columns))

X_train_full = train_wide.loc[:, feature_cols]
y_train_full = train_labels.to_numpy()

X_test = test_wide[feature_cols].copy()
y_test = test_labels.to_numpy()

# Impute missing values using training medians.
train_medians = X_train_full.median(axis=0)
X_train_full = X_train_full.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# Keep an internal validation split from the training cohort.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    train_size=0.7,
    random_state=1,
    stratify=y_train_full,
)

print(f"Using {len(feature_cols)} shared assay features")

Finally, we also create another held-out validation set from the training data.
This will hopefully help us make the best modeling decisions possible.

Thus we now have:
1. A training set to use for actually training the models.
2. A validation set derived from the same cohort to help us choose hyperparameters and make decisions.
3. A test set from another cohort that will be a gauge for how our final model would perform in the wild.


In [ ]:
# Keep an internal validation split from the training cohort.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    train_size=0.7,
    random_state=1,
    stratify=y_train_full,
)

print("Training matrix shape:", X_train.shape)
print("Internal cohort validation shape:", X_valid.shape)
print("External cohort test shape:", X_test.shape)

## Fit logistic regression

We'll start with logistic regression, which is likely a good fit for this many-features-but-few-samples paradigm.
Given this situation, we'll also test using L1 regularization to limit the number of proteins that are actually used by the model in the end.

Note that for both logistic regression and the support vector machine to come, we first "standardize" the data using sklearn's StandardScaler.
This calculates the mean and standard deviation for each feature in the training set, then subtracts that mean and divides by the standard deviation such that the mean for each feature is 0 and the standard deviation is 1.
This is critical for interpretability and also helps the models learn faster.

Before we train, let's set up a variable to track our performance experiments:


In [ ]:
log_reg_tracker = {"C": [], "train": [], "valid": []}

Now let's fit some logistic regression models.
Try changing the `LOGREG_C` variable below.
The `C` in scikit-learn's implementation of logistic regression is the inverse of the regularization strength.
It ranges between 0 and infinity, and values closer to 0 add stronger regularization.

Here, we're monitoring the area under the receiver operating characteristic curve (AUROC) as a metric for performance, which summarizes the trade-off between the true positive rate and false positive rate for our classification task.
If you have more questions about ROC curves, Will made [a video about it](https://www.youtube.com/watch?v=up7Bl0eeedk).

As you change values of `LOGREG_C`, the plot will update. What value should you choose? Make sure to set it to that value and run the cell before you move on!


In [ ]:
LOGREG_C = 1  # <- Try changing this!

log_reg = Pipeline(
    steps=[
        # Scales all of the features to have a mean of 0 and
        # a standard deviation of 1.
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                C=LOGREG_C,
                l1_ratio=0,
                class_weight="balanced",
                solver="liblinear",
            ),
        ),
    ]
)

log_reg.fit(X_train, y_train)

# Internal training cohort performance
log_score_train = log_reg.predict_proba(X_train)[:, 1]
log_auroc_train = roc_auc_score(y_train, log_score_train)

# Internal validation cohort performance
log_score_valid = log_reg.predict_proba(X_valid)[:, 1]
log_auroc_valid = roc_auc_score(y_valid, log_score_valid)

print("Internal training cohort AUROC:  ", log_auroc_train)
print("Internal validation cohort AUROC:", log_auroc_valid)

# Update our tracker too:
if LOGREG_C not in log_reg_tracker["C"]:
    log_reg_tracker["C"].append(LOGREG_C)
    log_reg_tracker["train"].append(log_auroc_train)
    log_reg_tracker["valid"].append(log_auroc_valid)

# Make a plot of the AUROC performance so far:
lr_auroc_df = pd.DataFrame(log_reg_tracker)

plt.figure()
sns.lineplot(
    data=lr_auroc_df, x="C", y="train", marker="o", label="Training Set"
)
sns.lineplot(
    data=lr_auroc_df, x="C", y="valid", marker="o", label="Validation Set"
)

plt.xlabel("C")
plt.xscale("log")
plt.ylabel("Logistic Regression AUROC")

## Fit support vector machine (SVM)

Now we'll try fitting a more flexible support vector machine model with a radial basis function (RBF) kernel.
This works similarly to our code above.

First, we'll set up a tracker for performance:


In [ ]:
svm_tracker = {"C": [], "train": [], "valid": []}

Like with logistic regression, try changing values of `SVM_C` below.

For the SVM, the `C` parameter in scikit-learn is the inverse of the "cost" hyperparameter.
The cost describes how lenient the model is when classifying samples on the wrong side of the margin.
Practically, it has a similar effect as the `C` above: it ranges between 0 and infinity, and lower values increase the strength of regularization.


In [ ]:
SVM_C = 1  # <- Try changing this! Tip: 1e-10 works as scientific notation.

svm = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            SVC(
                kernel="rbf",
                C=SVM_C,
                gamma="scale",
                class_weight="balanced",
            ),
        ),
    ]
)

svm.fit(X_train, y_train)

# Internal training cohort performance
svm_score_train = svm.decision_function(X_train)
svm_auroc_train = roc_auc_score(y_train, svm_score_train)

# Internal validation cohort performance
svm_score_valid = svm.decision_function(X_valid)
svm_auroc_valid = roc_auc_score(y_valid, svm_score_valid)

print("Internal training cohort AUROC:  ", svm_auroc_train)
print("Internal validation cohort AUROC:", svm_auroc_valid)

# Update our tracker too:
if SVM_C not in svm_tracker["C"]:
    svm_tracker["C"].append(SVM_C)
    svm_tracker["train"].append(svm_auroc_train)
    svm_tracker["valid"].append(svm_auroc_valid)

# Make a plot of the AUROC performance so far:
svm_auroc_df = pd.DataFrame(svm_tracker)

plt.figure()
sns.lineplot(
    data=svm_auroc_df, x="C", y="train", marker="o", label="Training Set"
)
sns.lineplot(
    data=svm_auroc_df, x="C", y="valid", marker="o", label="Validation Set"
)

plt.xlabel("C")
plt.xscale("log")
plt.ylabel("Support Vector Machine AUROC")
plt.show()

## STOP!

It's now time to choose your final logistic regression and SVM models.
Return to the cells above and set your values, then make sure you run the cell.
When you're ready, proceed to the next section.


## Evaluate on the Test Set from a New Cohort

Now that our models are locked in, we travel through time and suddenly have a new cohort of data to test our model on.
Below, we use our models to predict on this new cohort, calculate the AUROC, and plot the ROC curves.

How did you fare?


In [ ]:
log_score_test = log_reg.predict_proba(X_test)[:, 1]
log_auroc_test = roc_auc_score(y_test, log_score_test)

svm_score_test = svm.decision_function(X_test)
svm_auroc_test = roc_auc_score(y_test, svm_score_test)

# Plot external validation curves.
fpr_log, tpr_log, _ = roc_curve(y_test, log_score_test)
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_score_test)

plt.figure()
plt.plot(fpr_log, tpr_log, label=f"LogReg (AUC={log_auroc_test:0.3f})")
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC={svm_auroc_test:0.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("External Cohort ROC Curve")
plt.legend()

plt.tight_layout()
plt.show()

print("Logistic Regression AUROC:", log_auroc_test)
print("SVM AUROC:", svm_auroc_test)

Did it go well? If so, congratulations, you are one of the few!

This example is a perfect storm of a difficult situation:
1. There are too few examples in the training set for the number of features we want to model.
2. There are too few examples in the validation set to get a good idea of performance.
3. The test dataset is from a completely new cohort. If we don't have this data at the time of modeling, many things can happen that are unique to each cohort and not generalizable between them.
